*This is the notebook for the inference phase of this example*

## Feed Forward Network Demonstration
**Goal:** Predict the function $f(x, y) = e^{-(x^2 + y^2)}$

**Strategy:**
The training strategy was all in the file `model.ipynb`, the inference process is much simpler.
1. We build a neural network with architecture $2 - 16 - 16 - 1$. With reLU layer in between.
2. We will try to measure how good our model is with $101 \times 101$ points, from $(-10, -10)$ to $(10, 10)$, with step of $0.1$ in between.
4. We load our previous training result from the `.cache` file

This library works similarly to Pytorch, and I'm proud of that.

### 1. Import and Config

In [2]:
import random
import math
import lktorch as lk
import os

CURRENT_DIR = os.getcwd()
WEIGHT_PATH = os.path.join(CURRENT_DIR, "training_data", "model_weight.cache")

### 2. Generate and preprocess

In [3]:
private_test = []
for i in range(-200, 201):
    for j in range(-200, 201):
        private_test.append([i / 10, j / 10, math.exp(- (i / 10)**2 - (j / 10) ** 2)])

random.shuffle(private_test)

print("Done fetching data!")

Done fetching data!


### 3. Setting up the model

In [4]:

private_test_tensor = lk.Tensor(private_test)
input_data = private_test_tensor.Slice([0], [1])
output_data = private_test_tensor.Slice([2], [2])

model = lk.Sequential([
    lk.LinearLayer(2, 16),
    lk.reLU_Layer(),
    lk.LinearLayer(16, 16),
    lk.reLU_Layer(),
    lk.LinearLayer(16, 1),
    lk.Sigmoid_Layer(),
])


def calculate_loss(my_nn, input, output, LossFunction):
    tenso = my_nn(input)
    tenso = LossFunction(tenso, output)
    return lk.MeanAll(tenso).accessA([])

lk.deactivate_backprop()

### 4. Measuring the untrained network

In [ ]:
print("No training:")
my_output = model(input_data)
mse_loss = lk.MeanAll(lk.MSELoss(my_output, output_data))
mae_loss = lk.MeanAll(lk.MAELoss(my_output, output_data))
rmse_loss = lk.MeanAll(lk.RMSELoss(my_output, output_data))
print("MSE Loss: ", mse_loss)
print("MAE Loss: ", mae_loss)
print("RMSE Loss: ", rmse_loss)

No training:
MSE Loss:  0.223103
MAE Loss:  0.260402
RMSE Loss:  0.260403
Yes training:
MSE Loss:  0.223103
MAE Loss:  0.260402
RMSE Loss:  0.260403


### 5. Measuring the trained network

In [6]:
model.load_state_dict(lk.ReadFile(WEIGHT_PATH))

print("Yes training:")
my_output = model(input_data)
mse_loss = lk.MeanAll(lk.MSELoss(my_output, output_data))
mae_loss = lk.MeanAll(lk.MAELoss(my_output, output_data))
rmse_loss = lk.MeanAll(lk.RMSELoss(my_output, output_data))
print("MSE Loss: ", mse_loss)
print("MAE Loss: ", mae_loss)
print("RMSE Loss: ", rmse_loss)

Yes training:
MSE Loss:  0.000176852
MAE Loss:  0.000798615
RMSE Loss:  0.000845872


As you can see, the network is extremely accurate!